In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import string

In [25]:
df=pd.read_csv('IMDB Dataset.csv')

In [26]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [27]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

DATA CLEANING

CONVERTING TO LOWER CASE

In [28]:
df['review']=df['review'].str.lower()

REMOVING HTML TAGS

In [29]:
from bs4 import BeautifulSoup

def remove_html(text):
    return BeautifulSoup(str(text), "html.parser").get_text(" ")

df['review'] = df['review'].apply(remove_html)

REMOVING PUNCTUATION SIGN ETC

In [30]:
def remove_punc(text):
    return text.translate(str.maketrans('','',string.punctuation))

In [31]:
df['review']=df['review'].apply(remove_punc)

REMOVING NUMBERS

In [32]:
def remove_numb(text):
    new=""
    for i in text:
        if not i.isdigit():
            new=new+i
    return new 

df['review']=df['review'].apply(remove_numb)

In [33]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production the filming tec...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


IMPORTING NATURAL LANGUAGE TOOL KIT(NLTK) - FOR REMOVING STOP WORDS

In [34]:
import nltk 

In [35]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

In [36]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vansh\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vansh\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [37]:
stop_words=set(stopwords.words('english'))
len(stop_words)

198

In [38]:
df['review'].loc[1]

'a wonderful little production  the filming technique is very unassuming very oldtimebbc fashion and gives a comforting and sometimes discomforting sense of realism to the entire piece  the actors are extremely well chosen michael sheen not only has got all the polari but he has all the voices down pat too you can truly see the seamless editing guided by the references to williams diary entries not only is it well worth the watching but it is a terrificly written and performed piece a masterful production about one of the great masters of comedy and his life  the realism really comes home with the little things the fantasy of the guard which rather than use the traditional dream techniques remains solid then disappears it plays on our knowledge and our senses particularly with the scenes concerning orton and halliwell and the sets particularly of their flat with halliwells murals decorating every surface are terribly well done'

In [39]:
def remove_stop(text):
    words=text.split()
    cleaned=[]
    for i in words:
        if i not in stop_words:
            cleaned.append(i)
    return ' '.join(cleaned)


In [40]:
df['review']=df['review'].apply(remove_stop)

In [41]:
df.head()

,review,sentiment
0,one reviewers mentioned watching oz episode yo...,positive
1,wonderful little production filming technique ...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically theres family little boy jake thinks...,negative
4,petter matteis love time money visually stunni...,positive


In [42]:
df['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

CONVERTING SENTIMENT TO 0 & 1

In [43]:
df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

In [44]:
df.head()

,review,sentiment
0,one reviewers mentioned watching oz episode yo...,1
1,wonderful little production filming technique ...,1
2,thought wonderful way spend time hot summer we...,1
3,basically theres family little boy jake thinks...,0
4,petter matteis love time money visually stunni...,1


In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['review'], df['sentiment'], test_size=0.20, random_state=42)

In [46]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)

pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))

0.8601


In [47]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

nb2_model = MultinomialNB()
nb2_model.fit(X_train_tfidf,y_train)

MultinomialNB()

In [48]:
y_pred = nb2_model.predict(X_test_tfidf)

In [49]:
print(accuracy_score(y_test, y_pred))

0.8688


In [51]:
from sklearn.linear_model import LogisticRegression
logistic_model = LogisticRegression(max_iter=1000)
logistic_model.fit(X_train_tfidf,y_train)
log_pred = logistic_model.predict(X_test_tfidf)

In [52]:
print(accuracy_score(y_test,log_pred ))

0.8958


TESTING OWN 

In [53]:
review = ["This movie was absolutely amazing. The acting was brilliant and I loved every minute of it."]

In [55]:
review_tfidf = tfidf_vectorizer.transform(review)

In [56]:
prediction = logistic_model.predict(review_tfidf)

print(prediction)

[1]
